In [ ]:
## Steps for regression imputation 

# 1. Import related libraries
# 2. Load dataset
# 3. Remove problem causing columns
# 4. Encode the catagorical data to numeric format
# 5. Split data in two halves (with & without missing values)
# 6. Train the model on without missing values
# 7. Predict the values for with missing values
# 8. Replace the predicted column with missing values dataset
# 9. Concatenate the splited data
# 10. Decode the encoded data
# 11. Save the data in csv file

In [6]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import seaborn as sns
import pandas as pd
import numpy as np

In [7]:
df = sns.load_dataset("titanic")
df.isnull().sum().sort_values(ascending=False)

deck           688
age            177
embarked         2
embark_town      2
sex              0
pclass           0
survived         0
fare             0
parch            0
sibsp            0
class            0
adult_male       0
who              0
alive            0
alone            0
dtype: int64

In [8]:
## don't know why but removing deck column

df.drop("deck", axis=1, inplace=True)

df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True


In [9]:
## label encoding

from sklearn.preprocessing import LabelEncoder

column_to_encode = ['sex', 'embarked', 'who', 'class', 'embark_town', 'alive']
label_encoders = {}

for col in column_to_encode:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,1,22.0,1,0,7.2500,2,2,1,True,2,0,False
1,1,1,0,38.0,1,0,71.2833,0,0,2,False,0,1,False
2,1,3,0,26.0,0,0,7.9250,2,2,2,False,2,1,True
3,1,1,0,35.0,1,0,53.1000,2,0,2,False,2,1,False
4,0,3,1,35.0,0,0,8.0500,2,2,1,True,2,0,True


In [10]:
## split the data in two halves

df_with_missing = df[df['age'].isna()]
df_without_missing = df.dropna()

df_with_missing.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
5,0,3,1,NaN,0,0,8.4583,1,2,1,True,1,0,True
17,1,2,1,NaN,0,0,13.0000,2,1,1,True,2,1,True
19,1,3,0,NaN,0,0,7.2250,0,2,2,False,0,1,True
26,0,3,1,NaN,0,0,7.2250,0,2,1,True,0,0,True
28,1,3,0,NaN,0,0,7.8792,1,2,2,False,1,1,True


In [11]:
print(f"The shape of original dataset: {df.shape}")
print(f"The shape of dataset with missing values: {df_with_missing.shape}")
print(f"The shape of dataset without missing values: {df_without_missing.shape}")

The shape of original dataset: (891, 14)
The shape of dataset with missing values: (177, 14)
The shape of dataset without missing values: (714, 14)


In [12]:
print(df.columns)

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'embark_town', 'alive',
       'alone'],
      dtype='object')


In [13]:
## Model Training
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

X = df_without_missing.drop(["age"], axis=1)
y = df_without_missing['age']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_predict = rf_model.predict(X_test)

print(f"R2 score: {r2_score(y_test, y_predict)}")
print(f"MSE score: {mean_squared_error(y_test, y_predict)}")
print(f"MAE score: {mean_absolute_error(y_test, y_predict)}")

R2 score: 0.33769388288226154
MSE score: 122.79433625923292
MAE score: 8.666661815622195


In [14]:
df_with_missing.isnull().sum().sort_values(ascending=False)

age            177
survived         0
pclass           0
sex              0
sibsp            0
parch            0
fare             0
embarked         0
class            0
who              0
adult_male       0
embark_town      0
alive            0
alone            0
dtype: int64

In [15]:
## predict the actual missing values 

y_predict = rf_model.predict(df_with_missing.drop(['age'], axis=1))

print(y_predict)

[32.97658333 35.64221825 18.347      35.57148611 20.65142857 26.7619855
 36.648      18.63142857 21.80633333 33.55618169 31.06587652 35.90741667
 18.63142857 24.824      31.03       39.405      25.849      26.7619855
 31.06587652 19.41142857 31.06587652 31.06587652 26.7619855  26.27095821
 29.23514286 31.06587652 48.25650595 27.94       31.87071429 31.99628481
 30.015      20.85816667 33.755      60.19168831 26.00185714 26.24316667
 28.91733333 49.31       28.55277778 48.25650595 18.63142857 20.85816667
 33.78929167 26.7619855  26.63       32.01066667 28.22883333 28.55277778
 31.99628481 29.72904762 48.25650595 27.67733333 56.26333333 18.63142857
 34.65645944 60.44168831 39.405      35.7725     18.63142857 24.78266667
 34.305      31.06587652 31.602      20.85816667 25.296      36.97133333
 26.7619855  24.85777778 55.52       35.57148611 20.65142857 20.65142857
 35.90741667 18.001      18.63142857 39.47333333 26.7619855  30.16908333
 26.63       26.7619855  24.02805952 34.65645944 29.4

In [16]:
## put the predicted values in data with missing values

df_with_missing['age'] = y_predict
df_with_missing

C:\Users\hamza\AppData\Local\Temp\ipykernel_18016\4047783239.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_with_missing['age'] = y_predict


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
5,0,3,1,32.976583,0,0,8.4583,1,2,1,True,1,0,True
17,1,2,1,35.642218,0,0,13.0000,2,1,1,True,2,1,True
19,1,3,0,18.347000,0,0,7.2250,0,2,2,False,0,1,True
26,0,3,1,35.571486,0,0,7.2250,0,2,1,True,0,0,True
28,1,3,0,20.651429,0,0,7.8792,1,2,2,False,1,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859,0,3,1,26.966917,0,0,7.2292,0,2,1,True,0,0,True
863,0,3,0,26.243167,8,2,69.5500,2,2,2,False,2,0,False
868,0,3,1,24.816926,0,0,9.5000,2,2,1,True,2,0,True
878,0,3,1,26.761985,0,0,7.8958,2,2,1,True,2,0,True


In [17]:
df_with_missing.isnull().sum().sort_values(ascending=False)

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64

In [18]:
df_with_missing

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
5,0,3,1,32.976583,0,0,8.4583,1,2,1,True,1,0,True
17,1,2,1,35.642218,0,0,13.0000,2,1,1,True,2,1,True
19,1,3,0,18.347000,0,0,7.2250,0,2,2,False,0,1,True
26,0,3,1,35.571486,0,0,7.2250,0,2,1,True,0,0,True
28,1,3,0,20.651429,0,0,7.8792,1,2,2,False,1,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859,0,3,1,26.966917,0,0,7.2292,0,2,1,True,0,0,True
863,0,3,0,26.243167,8,2,69.5500,2,2,2,False,2,0,False
868,0,3,1,24.816926,0,0,9.5000,2,2,1,True,2,0,True
878,0,3,1,26.761985,0,0,7.8958,2,2,1,True,2,0,True


In [19]:
## merge both data sets

df_complete = pd.concat([df_with_missing, df_without_missing], axis=0)
df_complete.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
5,0,3,1,32.976583,0,0,8.4583,1,2,1,True,1,0,True
17,1,2,1,35.642218,0,0,13.0000,2,1,1,True,2,1,True
19,1,3,0,18.347000,0,0,7.2250,0,2,2,False,0,1,True
26,0,3,1,35.571486,0,0,7.2250,0,2,1,True,0,0,True
28,1,3,0,20.651429,0,0,7.8792,1,2,2,False,1,1,True


In [20]:
## decoding

for col in column_to_encode:
    le = label_encoders[col]
    df_complete[col] = le.inverse_transform(df[col])
    
df_complete.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
5,0,3,male,32.976583,0,0,8.4583,S,Third,man,True,Southampton,no,True
17,1,2,female,35.642218,0,0,13.0000,C,First,woman,True,Cherbourg,yes,True
19,1,3,female,18.347000,0,0,7.2250,S,Third,woman,False,Southampton,yes,True
26,0,3,female,35.571486,0,0,7.2250,S,First,woman,True,Southampton,yes,True
28,1,3,male,20.651429,0,0,7.8792,S,Third,man,False,Southampton,no,True


In [21]:
print(f"Shape of complete data {df_complete.shape}")

Shape of complete data (891, 14)


In [ ]:
# df_complete.to_csv("Titanic_With_Imputed_Values.csv", index=False)
# df_complete.to_excel("Titanic_With_Imputed_Values.xlsx", index=False)

In [ ]:
data = pd.read_csv("Titanic_With_Imputed_Values")
data.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,male,32.976583,0,0,8.4583,S,Third,man,True,Southampton,no,True
1,1,2,female,35.642218,0,0,13.0000,C,First,woman,True,Cherbourg,yes,True
2,1,3,female,18.347000,0,0,7.2250,S,Third,woman,False,Southampton,yes,True
3,0,3,female,35.571486,0,0,7.2250,S,First,woman,True,Southampton,yes,True
4,1,3,male,20.651429,0,0,7.8792,S,Third,man,False,Southampton,no,True
